# Data cleaning

Notebook objectives:

- reload the raw dataset;
- convert the `date` column to datetime;
- check missing values;
- handle missing values in `AWND`;
- investigate extreme values in `daily_consumption`;
- prepare a cleaner dataset for the next project steps.

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [52]:
data_path = "../data/electricity_consumption_based_weather_dataset.csv"

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])

df.head()

,date,AWND,PRCP,TMAX,TMIN,daily_consumption
0,2006-12-16,2.5,0.0,10.6,5.0,1209.176
1,2006-12-17,2.6,0.0,13.3,5.6,3390.460
2,2006-12-18,2.4,0.0,15.0,6.7,2203.826
3,2006-12-19,2.4,0.0,7.2,2.2,1666.194
4,2006-12-20,2.4,0.0,7.2,1.1,2225.748


In [53]:
missing_values = df.isna().sum()

missing_values

date                  0
AWND                 15
PRCP                  0
TMAX                  0
TMIN                  0
daily_consumption     0
dtype: int64

In [54]:
missing_percentage = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count" : missing_values,
    "missing_percentage" : missing_percentage

})

missing_summary

,missing_count,missing_percentage
date,0,0.00
AWND,15,1.05
PRCP,0,0.00
TMAX,0,0.00
TMIN,0,0.00
daily_consumption,0,0.00


In [55]:
df[df["AWND"].isna()]

,date,AWND,PRCP,TMAX,TMIN,daily_consumption
888,2009-05-23,NaN,0.0,22.8,15.6,1530.598
889,2009-05-24,NaN,0.0,28.3,15.6,1287.786
890,2009-05-25,NaN,0.0,27.2,17.2,1354.096
891,2009-05-26,NaN,0.0,18.9,11.7,1337.674
895,2009-05-30,NaN,0.0,25.0,15.6,1659.106
896,2009-05-31,NaN,0.0,27.2,13.9,956.890
921,2009-06-26,NaN,18.5,28.3,17.8,1277.322
922,2009-06-27,NaN,0.5,27.2,17.2,1709.052
923,2009-06-28,NaN,0.0,27.8,17.8,1059.394
1126,2010-01-18,NaN,1.3,10.0,2.8,1881.098


In [56]:
missing_awnd_dates = df.loc[df["AWND"].isna(), "date"]

missing_awnd_dates

888    2009-05-23
889    2009-05-24
890    2009-05-25
891    2009-05-26
895    2009-05-30
896    2009-05-31
921    2009-06-26
922    2009-06-27
923    2009-06-28
1126   2010-01-18
1127   2010-01-19
1133   2010-01-25
1373   2010-09-28
1375   2010-09-30
1376   2010-10-01
Name: date, dtype: datetime64[ns]

In [57]:
missing_awnd_dates.diff()

888         NaT
889      1 days
890      1 days
891      1 days
895      4 days
896      1 days
921     26 days
922      1 days
923      1 days
1126   204 days
1127     1 days
1133     6 days
1373   246 days
1375     2 days
1376     1 days
Name: date, dtype: timedelta64[ns]

In [58]:
df_clean = df.copy()

df_clean["AWND"] = df_clean["AWND"].interpolate(method="linear")

df_clean.isna().sum()

date                 0
AWND                 0
PRCP                 0
TMAX                 0
TMIN                 0
daily_consumption    0
dtype: int64

## Missing value handling decision

`AWND` is the only column with missing values: 15 missing observations, representing about 1.05% of the dataset.

The missing values appear in small time-based groups rather than being fully isolated. Since `AWND` is a daily weather measurement, we use linear interpolation to estimate missing values from neighboring days.

The original DataFrame `df` is kept unchanged, and the cleaned version is stored in `df_clean`.

In [59]:
df_clean["daily_consumption"].describe()

count    1433.000000
mean     1561.078061
std       606.819667
min        14.218000
25%      1165.700000
50%      1542.650000
75%      1893.608000
max      4773.386000
Name: daily_consumption, dtype: float64

In [60]:
low_consumption = df_clean.sort_values("daily_consumption").head(10)

low_consumption

,date,AWND,PRCP,TMAX,TMIN,daily_consumption
909,2009-06-13,1.9,3.3,22.8,16.1,14.218
133,2007-04-28,1.8,0.0,21.7,12.2,22.738
1338,2010-08-22,2.9,42.9,26.7,20.6,93.338
1372,2010-09-25,3.2,0.0,31.7,19.4,106.006
1187,2010-03-20,2.2,0.0,23.3,12.2,131.732
617,2008-08-25,2.6,0.0,29.4,19.4,250.298
615,2008-08-23,1.7,0.0,27.2,18.9,254.642
616,2008-08-24,2.2,0.0,27.8,20.0,254.716
618,2008-08-26,2.7,0.0,26.7,16.1,257.564
619,2008-08-27,2.3,0.0,26.1,16.1,258.114


In [61]:
q1 = df_clean["daily_consumption"].quantile(0.25)
q3 = df_clean["daily_consumption"].quantile(0.75)

iqr = q3 -q1

lower_bound = q1 -1.5 * iqr
upper_bound = q3 + 1.5 * iqr

q1, q3, iqr, lower_bound, upper_bound

(np.float64(1165.7),
 np.float64(1893.608),
 np.float64(727.9079999999999),
 np.float64(73.83800000000019),
 np.float64(2985.47))

In [62]:
consumption_outliers = df_clean[
    (df_clean["daily_consumption"]< lower_bound) |
    (df_clean["daily_consumption"] > upper_bound)

]

consumption_outliers.sort_values("daily_consumption")

,date,AWND,PRCP,TMAX,TMIN,daily_consumption
909,2009-06-13,1.9,3.3,22.8,16.1,14.218
133,2007-04-28,1.8,0.0,21.7,12.2,22.738
39,2007-01-24,2.6,0.0,4.4,0.0,2987.854
29,2007-01-14,1.9,2.3,7.8,5.0,3007.816
699,2008-11-15,2.9,18.8,19.4,14.4,3031.628
43,2007-01-28,1.7,1.3,5.0,-1.1,3090.204
376,2007-12-28,2.1,0.5,9.4,4.4,3113.052
783,2009-02-07,1.7,0.0,10.0,-2.2,3131.536
35,2007-01-20,5.1,0.3,0.0,-6.1,3133.732
689,2008-11-05,4.1,7.4,17.8,12.2,3134.930


In [63]:
print(f"Q1: { q1:.2f} ")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")

Q1: 1165.70 
Q3: 1893.61
IQR: 727.91
Lower bound: 73.84
Upper bound: 2985.47


In [64]:
low_consumption_outliers = df_clean[df_clean["daily_consumption"] < lower_bound]
high_consumption_outliers = df_clean[df_clean["daily_consumption"] > upper_bound]

len(low_consumption_outliers), len(high_consumption_outliers)



(2, 32)

In [65]:
low_consumption_outliers.sort_values("daily_consumption")

,date,AWND,PRCP,TMAX,TMIN,daily_consumption
909,2009-06-13,1.9,3.3,22.8,16.1,14.218
133,2007-04-28,1.8,0.0,21.7,12.2,22.738


In [66]:
df_clean["is_low_consumption_outlier"] = df_clean["daily_consumption"] < lower_bound

df_clean["is_low_consumption_outlier"].sum()

np.int64(2)

## Low consumption outlier decision

The IQR method identifies 2 unusually low consumption values and 32 unusually high values.

The high values mostly correspond to winter periods and are consistent with the seasonal pattern observed during the exploration step. Therefore, they are kept in the dataset.

The 2 very low consumption values are much farther from the typical consumption range and may represent abnormal or incomplete measurements. Instead of removing them immediately, we flag them with `is_low_consumption_outlier` so that later modeling steps can decide whether to exclude them or treat them separately.

In [67]:
df_model_candidate = df_clean.copy()

df_model_candidate.head()

,date,AWND,PRCP,TMAX,TMIN,daily_consumption,is_low_consumption_outlier
0,2006-12-16,2.5,0.0,10.6,5.0,1209.176,False
1,2006-12-17,2.6,0.0,13.3,5.6,3390.460,False
2,2006-12-18,2.4,0.0,15.0,6.7,2203.826,False
3,2006-12-19,2.4,0.0,7.2,2.2,1666.194,False
4,2006-12-20,2.4,0.0,7.2,1.1,2225.748,False


In [68]:
output_path = "../data/processed/electricity_weather_cleaned.csv"

df_model_candidate.to_csv(output_path, index=False)

pd.read_csv(output_path).head()

,date,AWND,PRCP,TMAX,TMIN,daily_consumption,is_low_consumption_outlier
0,2006-12-16,2.5,0.0,10.6,5.0,1209.176,False
1,2006-12-17,2.6,0.0,13.3,5.6,3390.460,False
2,2006-12-18,2.4,0.0,15.0,6.7,2203.826,False
3,2006-12-19,2.4,0.0,7.2,2.2,1666.194,False
4,2006-12-20,2.4,0.0,7.2,1.1,2225.748,False


## Cleaned dataset export

The cleaned dataset is exported to `data/processed/electricity_weather_cleaned.csv`.

This version includes:

- interpolated missing values in `AWND`;
- the original weather and consumption columns;
- a boolean flag named `is_low_consumption_outlier` to identify unusually low consumption values.

The raw dataset remains unchanged in the `data/` folder.